In [2]:
!pip install findspark

import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Cross_Join").getOrCreate()

In [3]:
departamentos = spark.read.parquet("./data/departamentos")
departamentos.show()

+---+-----------+
| id|nombre_dpto|
+---+-----------+
| 31|     letras|
| 33|    derecho|
| 34| matemática|
| 35|informática|
+---+-----------+



In [4]:
empleados = spark.read.parquet("./data/empleados")
empleados.show()

+------+--------+
|nombre|num_dpto|
+------+--------+
|  Luis|      33|
| Katia|      33|
|  Raul|      34|
| Pedro|       0|
| Laura|      34|
|Sandro|      31|
+------+--------+



# Producto Cartesiano: `crossJoin()` (Cross Join)

En términos de sintaxis, este tipo de unión es el más simple de implementar debido a que **no requiere una expresión de unión (*join expression*)** ni evaluar ninguna condición de igualdad entre columnas.

---

## ⚠️ Nota de Advertencia: Un Comportamiento Peligroso

A pesar de su simplicidad, el *Cross Join* es una operación altamente peligrosa en entornos de Big Data si no se maneja con precaución. Su comportamiento consiste en emparejar **cada fila del conjunto de datos de la izquierda con todas y cada una de las filas del conjunto de datos de la derecha**, lo que matemáticamente se conoce como un **producto cartesiano**.



### Impacto en el Volumen de Datos:
El tamaño del DataFrame resultante será exactamente el **producto matemático** del tamaño de ambos conjuntos de datos ($N \times M$).

* **Ejemplo Práctico:** Si tienes dos conjuntos de datos relativamente pequeños, cada uno con solo **1,024 registros**, el resultado de la unión generará:
  $$1,024 \times 1,024 = 1,048,576 \text{ filas (más de 1 Millón de registros)}$$

Si aplicaras esto por error a datasets reales que contienen millones de filas, la explosión de datos saturaría instantáneamente la memoria RAM de los nodos, deteniendo tu pipeline por un error de tipo *Out of Memory* (OOM).

---

## Buenas Prácticas: Uso Explícito en PySpark

Debido a este riesgo latente, Spark cuenta con una medida de seguridad. Para evitar que ejecutes un producto cartesiano por accidente (por ejemplo, al omitir la condición en un `.join()` tradicional), la API de Spark exige que esta operación se invoque mediante un método dedicado y explícito llamado **`.crossJoin()`**:

```python
# Forma explícita y segura de ejecutar un producto cartesiano en PySpark
df_resultado = df_izquierda.crossJoin(df_derecha)

In [5]:
from pyspark.sql.functions import col

In [7]:
df = empleados.crossJoin(departamentos)

In [8]:
df.show()

+------+--------+---+-----------+
|nombre|num_dpto| id|nombre_dpto|
+------+--------+---+-----------+
|  Luis|      33| 31|     letras|
|  Luis|      33| 33|    derecho|
|  Luis|      33| 34| matemática|
|  Luis|      33| 35|informática|
| Katia|      33| 31|     letras|
| Katia|      33| 33|    derecho|
| Katia|      33| 34| matemática|
| Katia|      33| 35|informática|
|  Raul|      34| 31|     letras|
|  Raul|      34| 33|    derecho|
|  Raul|      34| 34| matemática|
|  Raul|      34| 35|informática|
| Pedro|       0| 31|     letras|
| Pedro|       0| 33|    derecho|
| Pedro|       0| 34| matemática|
| Pedro|       0| 35|informática|
| Laura|      34| 31|     letras|
| Laura|      34| 33|    derecho|
| Laura|      34| 34| matemática|
| Laura|      34| 35|informática|
+------+--------+---+-----------+
only showing top 20 rows


In [9]:
# solo nos esta mostrando 20 filas por la accion show

In [12]:
# 6 empleados x 4 departamentos = 24 filas

df.count()

24

In [13]:
# vamos a tener cuidado con este tipo de join porque la volumetria podria convertirse en un problema en determinados casos